# Module 5a - Linear Subspaces: SVD, PCA, Eigen-Cells, Robust PCA & LDA

High-dimensional life-science measurements -- gene-expression matrices, imaging stacks --
are almost never as complex as their raw dimensionality suggests; the interesting biology
usually lives on a low-dimensional surface inside a much larger measurement space. Part 1 of
this two-part module develops the **linear** machinery that finds that surface: the singular value
decomposition and **PCA**, the **eigen-cells** that are PCA applied to images, **robust PCA**
(low-rank signal plus sparse corruption), and **LDA** (the supervised projection when labels are
available). Part 2 (Module 5b) goes beyond linear subspaces to independent-source separation and
nonlinear manifolds.

**Reading.** Kutz, *Data-Driven Modeling & Scientific Computation*, 2nd ed., Chapter 15 (the
singular value decomposition and its use in PCA) and Chapter 18, section 2 (linear discriminant analysis). Read them for the derivations; everything below is
explained in our own terms and run against our own fixtures.

**Learning goals.**

- Connect the SVD to principal component analysis and read explained-variance ratios off a scree plot.
- Apply the same SVD to images as **eigen-cells**, and use a compact eigen-basis to reconstruct and *recognize* cells.
- Separate a matrix into **low-rank structure plus a sparse corruption** with robust PCA.
- Use **linear discriminant analysis (LDA)** for the supervised projection that separates labeled classes, and see where it beats PCA.
- Close every analysis with an explicit claim-and-limitations statement.

## Setup

We seed all random number generators and apply the course plotting style so the figures below are
deterministic and reproducible from a cold kernel.

```{admonition} Which paradigm?
:class: note
**Data-driven.** SVD and PCA make no mechanistic commitment: hand them a data matrix and they
return the directions of greatest variance, letting structure emerge inductively from the numbers
alone. **Eigen-cells** are the same move on images -- every cell crop written as a weighted sum of a
few shared image-space patterns. **Robust PCA** stays inductive but asserts one structural prior --
that the matrix is low-rank plus sparse -- which is exactly what lets it quarantine gross outliers an
ordinary SVD would smear. **LDA** adds labels: it finds the projection that best *separates* known
classes, beating PCA exactly when the class signal lies off the high-variance axes. Reach for these
linear methods when you trust your measurements but have no mechanism to deduce from -- and let a
known-ground-truth fixture, not faith, set how far to trust the result. Module 8 rediscovers this same
PCA subspace with a linear autoencoder.
```

In [ ]:
# Colab setup: install the ddm4bio course library (+ this lesson's data deps).
# No-op when ddm4bio is already importable (e.g. the course-site build), so
# this cell is safe everywhere. It is hidden from the rendered site via the
# "remove-cell" tag, but runs when this notebook is opened in Google Colab.
try:
    import ddm4bio  # noqa: F401
except ModuleNotFoundError:
    %pip install -q "ddm4bio @ git+https://github.com/symbiont-ai/ddm4bio.git" scanpy anndata umap-learn
    import ddm4bio  # noqa: F401

In [ ]:
import numpy as np

import ddm4bio
from ddm4bio import seed_everything
from ddm4bio.viz.style import set_style

seed_everything()
set_style()

print(f"ddm4bio version: {ddm4bio.__version__}")

## 1. PCA via the SVD

Principal component analysis rotates the coordinate axes of a dataset so that
the first axis points along the direction of greatest variance, the second along
the greatest remaining variance orthogonal to the first, and so on. Numerically
this is just the singular value decomposition of the mean-centered data matrix:
the right singular vectors are the principal axes, and the squared singular
values are proportional to the variance captured along each axis.

To make PCA concrete we point it at real single-cell data. The PBMC3k dataset
is a classic 10x Genomics assay of peripheral-blood mononuclear cells from a
healthy donor: thousands of cells, each a sparse vector of gene counts. We pull
it through the course data layer, which fetches the genuine 10x matrix when it
can and otherwise returns a structurally identical synthetic single-cell matrix
so the analysis runs anywhere. The provenance line printed below tells you which
one you received; the analysis is written to work for either.

In [ ]:
from ddm4bio.datasets import get_dataset
from ddm4bio.methods.decomposition import explained_variance_ratio, pca_reduce

ds = get_dataset("pbmc3k")
payload = ds.payload

# get_dataset returns a real AnnData (with .X) or a labelled fallback dict.
if hasattr(payload, "X"):
    counts = payload.X
    labels = None
    obs = getattr(payload, "obs", None)
    if obs is not None:
        for col in ("cell_type", "cell_types", "louvain", "leiden", "bulk_labels"):
            if col in obs:
                labels = np.asarray(obs[col])
                break
else:
    counts = payload["counts"]
    labels = np.asarray(payload["labels"])

# 10x data ships as a sparse matrix; densify for the linear algebra below.
counts = np.asarray(counts.toarray() if hasattr(counts, "toarray") else counts, dtype=float)

print(f"[pbmc3k] source={ds.source}: {ds.provenance}")
groups = "unlabelled" if labels is None else f"{np.unique(labels).size} label groups"
print(f"expression matrix: {counts.shape[0]} cells x {counts.shape[1]} genes ({groups})")

Raw UMI counts are heavy-tailed and vary in sequencing depth from cell to cell,
so we apply the standard single-cell transform before any linear algebra:
normalize each cell to a common library size, then take `log1p`. We then keep
the most variable genes, which concentrates the biological signal and keeps the
full-width real matrix (tens of thousands of genes) tractable.

In [ ]:
# Library-size normalize, then log1p-compress the counts.
library = counts.sum(axis=1, keepdims=True)
library[library == 0] = 1.0
target = float(np.median(counts.sum(axis=1)))
log_counts = np.log1p(counts / library * target)

# Restrict to the top-variance genes (all of them when the matrix is already small).
n_keep = min(1000, log_counts.shape[1])
top_var = np.argsort(log_counts.var(axis=0))[::-1][:n_keep]
expr = log_counts[:, top_var]

scores = pca_reduce(expr, n_components=2)
evr = explained_variance_ratio(expr)

print(f"kept {expr.shape[1]} high-variance genes; PCA scores {scores.shape}")
print(f"leading explained-variance ratios: {np.round(evr[:5], 4)}")

Real expression data does not collapse to an exact low rank the way a clean
synthetic fixture does; instead the scree curve decays smoothly, and the "elbow"
is a judgement call about where genuine structure fades into the long tail of
biological and technical noise. We plot only the leading components, since the
tail is a near-flat noise floor.

In [ ]:
from ddm4bio.viz.plots import scree_plot

n_show = min(20, evr.size)
ax = scree_plot(evr[:n_show])
ax.figure;  # end the cell with the Figure so it renders in the notebook output

Projecting the cells onto their first two principal components gives the
familiar single-cell scatter. When the payload carries cell labels we colour by
them; the real 10x matrix ships unlabelled, so the same plot then renders as a
single-colour cloud whose structure we read from the loadings instead.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5.5, 4.5))
if labels is not None:
    for g in np.unique(labels):
        sel = labels == g
        ax.scatter(scores[sel, 0], scores[sel, 1], s=12, label=str(g))
    ax.legend(title="label", fontsize=8, loc="best")
else:
    ax.scatter(scores[:, 0], scores[:, 1], s=12)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("PBMC cells in principal-component space")
fig;

Finally we quantify how much of each leading component lines up with the provided
grouping, using the correlation ratio (eta), whose square eta-squared is the fraction of a
component's spread (variance) explained by label membership; eta itself is the square root
of that fraction -- a number in `[0, 1]`.

In [ ]:
def label_separation(values, group_labels):
    """Correlation ratio (eta) of a 1-D score against categorical labels."""
    values = np.asarray(values, dtype=float)
    grand = values.mean()
    total = ((values - grand) ** 2).sum()
    if total == 0.0:
        return 0.0
    between = sum(
        int((group_labels == g).sum()) * (values[group_labels == g].mean() - grand) ** 2
        for g in np.unique(group_labels)
    )
    return float(np.sqrt(between / total))


captured = float(evr[:2].sum())
if labels is not None:
    eta1 = label_separation(scores[:, 0], labels)
    eta2 = label_separation(scores[:, 1], labels)
    print(f"QC note ({ds.source} data): the top two PCs capture {captured:.1%} of "
          f"variance; label separation is eta(PC1)={eta1:.2f}, eta(PC2)={eta2:.2f}.")
else:
    print(f"QC note ({ds.source} data): the top two PCs capture {captured:.1%} of "
          "variance; this payload ships without cell labels, so the leading axes "
          "must be read from their gene loadings, not a provided grouping.")

**QC note.** On the labelled fallback the leading component already pulls the
synthetic cell types apart (a high eta), and a handful of components carry most
of the variance -- exactly the low-dimensional structure PCA is meant to
surface. On the real unlabelled matrix we instead lean on the scree shape and
the gene loadings, and we resist over-reading components buried in the noise
tail.


## 2. Eigen-cells: the same SVD, now on images

Section 1 found the principal directions of an *expression* matrix. Nothing about the SVD cares that
the rows were cells and the columns were genes -- feed it a matrix whose rows are **images** and its
directions become shared *image-space* patterns, the **eigen-cells** (the biomedical cousin of the
classic *eigenfaces*). Every cell crop is then a weighted sum of a few eigen-cells, which we use to
compress, reconstruct, and finally *recognize* cells from a compact basis.

**A Module 1 callback.** Module 1 read *conditioning* off the singular values: because a solve
divides by them, the smallest singular values amplify noise the most, with the damage bounded by
the condition number $\kappa = \sigma_{\max}/\sigma_{\min}$. That lesson carries straight into
truncation here. The small-$\sigma$ modes are the untrustworthy ones -- and they are the same
low-variance tail we *drop* (rather than divide by) when we truncate the eigen-basis in a moment.

### From pixels to an image library

The same tool -- the SVD -- now meets a *library* of images. Here the library is **real**: we pull
BloodMNIST -- peripheral-blood-cell microscopy crops from MedMNIST v2 -- through the course data
layer, `get_dataset("bloodmnist")`. Each crop is a small RGB image carrying an integer cell-type
label. We convert every crop to grayscale (averaging the colour channels) and flatten it to a
vector, using the **whole** training partition to build the basis (and the official test
partition to score, minus the handful of test crops byte-identical to a training image that the recognition step below drops as a leakage guard). The singular values of this library will play the very two roles we just
saw -- ranking directions by variance, and telling us which ones to trust.

In [ ]:
from ddm4bio.datasets import get_dataset

ds = get_dataset("bloodmnist", seed=0)
print(f"Data source : {ds.source}")
print(f"Provenance  : {ds.provenance}")

# BloodMNIST ships an OFFICIAL train / validation / test split (a 7:1:2 partition), so we use
# it rather than re-splitting. We use the WHOLE training partition to build the eigen-basis and
# the classifier, and score on the official test partition -- Section 5 later drops the few test crops
# byte-identical to a training image as a leakage guard. The
# colour channels are averaged to grayscale.
def to_grayscale(imgs):
    gray = imgs.mean(axis=-1)                              # (n, H, W) grayscale
    return gray, gray.reshape(gray.shape[0], -1).astype(float)


images, X = to_grayscale(ds.payload["train_images"])
y = ds.payload["train_labels"].ravel()
_, X_test = to_grayscale(ds.payload["test_images"])
y_test = ds.payload["test_labels"].ravel()
img_h, img_w = images.shape[1], images.shape[2]
classes = np.unique(np.concatenate([y, y_test]))
n_classes = classes.size

print(f"Training library : {X.shape[0]} images of {img_h}x{img_w} pixels ({n_classes} cell classes)")
print(f"Official test set: {X_test.shape[0]} held-out images (used only to score)")
print(f"Flattened feature matrix X: {X.shape} (samples x pixels)")

# BloodMNIST publishes named cell types (MedMNIST v2; Acevedo et al., 2020). The loader
# attaches them as `ds.labels` for the real data; the offline fallback is synthetic, so
# `ds.labels` is None there and we fall back to generic class ids.
def class_label(i):
    return ds.labels[int(i)] if ds.labels is not None else f"class {int(i)}"

A quick look at the cell types the recognizer has to work with -- one representative crop per
class (the *medoid*: the crop whose pixels are closest to its class mean, so it is the most
typical example rather than an arbitrary one). Even these representatives are low-resolution,
variable, and noisy.

In [ ]:
import matplotlib.pyplot as plt

# One representative crop per cell type: the class medoid -- the crop whose pixels are closest
# to that class's mean image, so it is the most typical example rather than an arbitrary one.
ncols = 4
nrows = int(np.ceil(n_classes / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(2.0 * ncols, 2.3 * nrows))
for i, ax in enumerate(np.atleast_1d(axes).ravel()):
    if i < n_classes:
        idx = np.where(y == classes[i])[0]
        if idx.size:
            medoid = idx[np.argmin(np.linalg.norm(X[idx] - X[idx].mean(axis=0), axis=1))]
            ax.imshow(images[medoid], cmap="gray_r")
            ax.set_title(class_label(classes[i]), fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"One representative crop per cell type ({n_classes} classes, {img_h}x{img_w} pixels)")
fig;

Before modeling, one look at the **class balance** of the training library. The eight cell
types are not equally represented -- worth keeping in mind when we read the 1-NN confusion
matrix later, since a nearest-neighbour rule leans toward the crowded classes and struggles
on the rare ones.

In [ ]:
train_classes, train_counts = np.unique(y, return_counts=True)
order = np.argsort(train_counts)[::-1]                     # most common first
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(range(len(order)), train_counts[order], color='0.6', edgecolor='white')
ax.bar_label(bars, padding=2, fontsize=8)
ax.set_xticks(range(len(order)))
ax.set_xticklabels([class_label(train_classes[i]) for i in order],
                   rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Training images')
ax.set_title(f'Class balance of the {y.size}-image training library')
ax.margins(y=0.12)
fig;

### Building the eigen-image basis

The eigen-image recipe is exactly PCA on the image library:

1. **Mean-center** -- subtract the average image so the basis describes
   *deviations* from the mean, not the mean itself.
2. **Take principal components** -- the right singular vectors of the centered
   data matrix are the eigen-images: orthogonal full-resolution pixel patterns (one weight per
   image pixel), ordered by how much library variance each explains.
3. **Project** -- every image becomes a short vector of coordinates in this
   basis (its PCA scores).

Because PCA is a *deterministic* linear algebra operation (an SVD), its "ground
truth" is self-checking: the explained-variance ratios are guaranteed to be
non-negative and to sum to one, and the top modes must reconstruct the data
better than any other orthogonal basis of the same size. We verify those
invariants explicitly rather than take them on faith.

In [ ]:
# Mean image and centered library.
mean_image = X.mean(axis=0)
X_centered = X - mean_image

# The eigen-images ARE the principal components: the right singular vectors of the centered
# library. NumPy's SVD returns them directly, ordered by how much variance each explains.
_, singular_values, vt = np.linalg.svd(X_centered, full_matrices=False)
evr = singular_values**2 / np.sum(singular_values**2)   # explained-variance ratio per mode

# How many independent directions can the data hold? The rank of a mean-centered matrix is
# capped by BOTH counts: rank <= min(n_samples - 1, n_pixels) -- centering costs one degree
# of freedom. With many more images than pixels the library is PIXEL-limited (rank = pixels);
# with fewer images than pixels it is SAMPLE-limited (rank = n_samples - 1, and that last
# centering direction is a numerically-zero artifact). We count the modes that carry signal
# with the tolerance rule np.linalg.matrix_rank uses, from the singular values already in hand.
rank_tol = singular_values[0] * max(X_centered.shape) * np.finfo(singular_values.dtype).eps
effective_rank = int(np.sum(singular_values > rank_tol))

print(f"Explained-variance ratios sum to 1: {np.isclose(evr.sum(), 1.0)}")
print(f"All ratios non-negative and non-increasing: "
      f"{np.all(evr >= 0) and np.all(np.diff(evr) <= 1e-12)}")
print(f"Variance captured by mode 1 alone : {evr[0]:.1%}")
print(f"Variance captured by top 10 modes : {evr[:10].sum():.1%}")
_n, _p = X_centered.shape
_regime = "pixel-limited" if _n - 1 >= _p else "sample-limited (centering kills one)"
print(f"Effective rank (modes above tol)  : {effective_rank} = min(n-1, pixels) = min({_n - 1}, {_p})  [{_regime}]")

The scree curve shows how fast the explained variance decays. Unlike the sharp
rank-2 elbow of a purely synthetic fixture, real image data has a *gentle*
shoulder: a handful of modes dominate, but a long tail of small modes carries
finer, lower-variance variation. What that tail *contains* is not settled by the
spectrum alone -- some may be class-relevant morphology, some may be nuisance
variation (illumination, staining, segmentation) or noise; the plot tells us only
that each of these modes carries little variance.

In [ ]:
from ddm4bio.viz.plots import scree_plot

ax = scree_plot(evr[:20])       # first 20 modes; the tail is a slow decay to zero
ax.set_title("Scree plot: explained variance of the top 20 eigen-images")
ax.figure;

Every reconstruction in this section starts from one picture: the **mean cell**, the
pixel-wise average of the training library. PCA writes each image as this mean *plus* a
weighted sum of the eigen-images that follow, so it is worth seeing on its own first.

In [ ]:
fig, ax = plt.subplots(figsize=(3.3, 3.3))
im = ax.imshow(mean_image.reshape(img_h, img_w), cmap='gray_r')
ax.set_xticks([]); ax.set_yticks([])
ax.set_title('Mean cell (library average)')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='mean intensity')
fig;

Now the eigen-images themselves -- the *signed directions of variation* in image
space, not template or prototype cells. Each is a full-resolution pixel pattern
reshaped back to the image grid. The first few look like smooth blobs that capture
gross cell shape and brightness; later ones encode progressively finer,
higher-frequency contrasts. Any library image is a weighted sum of the mean image
plus these patterns. One caveat on reading them: each mode's overall sign is an
arbitrary SVD convention -- flip a pattern's light and dark together and flip its
coordinate for every image, and nothing observable changes -- and it can differ
across linear-algebra backends. Treat the *structure* of each eigen-image as
meaningful, not its polarity.

In [ ]:
n_show = 8
eigen_images = vt[:n_show]                     # top-n signed variation patterns (rows of Vt)
vmax = float(np.abs(eigen_images).max())       # ONE scale, symmetric about zero, for all panels
ncols = 4
nrows = int(np.ceil(n_show / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(2.1 * ncols, 2.35 * nrows),
                         constrained_layout=True)
for i, ax in enumerate(axes.ravel()):
    if i < n_show:
        im = ax.imshow(eigen_images[i].reshape(img_h, img_w),
                       cmap='RdBu_r', vmin=-vmax, vmax=vmax)   # symmetric -> 0 maps to white
        ax.set_title(f'PC {i + 1}   ({evr[i]:.1%})', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Top 8 eigen-images ('eigen-cells'), each labelled with its explained variance")
fig.colorbar(im, ax=axes, shrink=0.85, fraction=0.05, pad=0.02,
             label='signed loading  (blue < 0 < red)')
fig;

### Reconstruction error vs. number of modes

How many eigen-images do we actually need? Project each image onto the top $k$ modes,
reconstruct it, and measure the error. As $k$ grows the reconstruction tightens; the useful
question is where the curve flattens -- the point past which extra modes buy little fidelity.
Keeping the top $k$ and discarding the rest drops exactly the small-$\sigma$ tail Module 1
flagged: it carries little variance *and* the smallest, least-trustworthy singular values, so
letting it go costs almost no fidelity and quietly suppresses noise (and, were we solving rather
than reconstructing, it is the truncation that tames the ill-conditioning). We report a single
**global (Frobenius) relative error** over the whole library,
$\lVert X_c - \hat{X}_c\rVert_F / \lVert X_c\rVert_F$ -- one number for the entire
sample-by-pixel matrix, *not* an average of per-image errors -- and overlay the
variance-captured milestones (90/95/99%). Because it is the global Frobenius error, it obeys an
*exact* identity with the variance spectrum, which we check below; we then look at how the error
is distributed across individual images.

In [ ]:
def reconstruct_with_k(X_centered, vt, k):
    """Project onto the top-k eigen-images and map back to pixel space."""
    basis = vt[:k]                      # (k, n_pixels)
    scores = X_centered @ basis.T       # (n_samples, k)
    return scores @ basis               # (n_samples, n_pixels), centered reconstruction


def global_frobenius_error(reference, approx):
    """One relative error for the WHOLE sample-by-pixel matrix: ||ref - approx||_F / ||ref||_F.

    This is a global (energy-weighted) Frobenius error, not the average of per-image relative
    errors -- high-norm images count for more. The next cell looks at the per-image spread.
    """
    return float(np.linalg.norm(reference - approx) / np.linalg.norm(reference))


# Variance milestones first, so the mode sweep and its plot span the real basis.
cum_evr = np.cumsum(evr)


def modes_for(threshold):
    return int(np.searchsorted(cum_evr, threshold) + 1)


k90, k95, k99 = modes_for(0.90), modes_for(0.95), modes_for(0.99)
n_modes = effective_rank            # the modes that carry signal (Section 2)

# A sweep that spans the whole basis -- adapting to the real 28x28 data (hundreds of modes)
# and the 8x8 offline fallback alike -- and always includes the variance milestones.
base = [1, 2, 4, 8, 16, 32, 64, 128, 256]
k_values = sorted({k for k in base if k < n_modes} | {k90, k95, k99, n_modes})
errors = [global_frobenius_error(X_centered, reconstruct_with_k(X_centered, vt, k))
          for k in k_values]

# Self-check: for the GLOBAL Frobenius error, reconstruction error is not merely correlated
# with explained variance -- it equals the square root of the leftover variance exactly,
#     ||X_c - X_c^(k)||_F / ||X_c||_F  ==  ||s[k:]|| / ||s||.
# We form the right-hand side from the TAIL of the singular values. (The tempting form
# sqrt(1 - cum_evr[k-1]) is algebraically identical but loses every digit to catastrophic
# cancellation near full rank -- 1 minus a sum that already equals ~1 -- so we avoid
# it here.)
s_norm = np.linalg.norm(singular_values)
tail_error = [float(np.linalg.norm(singular_values[k:]) / s_norm) for k in k_values]
assert np.allclose(errors, tail_error, rtol=0, atol=1e-12), \
    "reconstruction error must equal sqrt(leftover variance)"

print(f"Modes to reach 90% variance: {k90}")
print(f"Modes to reach 95% variance: {k95}")
print(f"Modes to reach 99% variance: {k99}  (of {n_modes} effective modes)")
print("Self-check passed: global Frobenius error == sqrt(leftover variance) for every k.")
for k, e in zip(k_values, errors):
    print(f"  k={k:3d}:  global Frobenius reconstruction error = {e:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_values, errors, marker="o", linewidth=1.5)
ax.set_xscale("log")
for (k, name), y_lab in zip([(k90, "90%"), (k95, "95%"), (k99, "99%")], [0.62, 0.40, 0.62]):
    ax.axvline(k, color="0.6", linestyle="--", linewidth=1)
    ax.text(k, y_lab, f"{name}\n(k={k})", fontsize=8, color="0.35", ha="center",
            backgroundcolor="white")
ax.set_xlabel("Number of eigen-images (k, log scale)")
ax.set_ylabel("Global (Frobenius) reconstruction error")
ax.set_title("Reconstruction error falls as the eigen-basis grows")
fig;

That curve is one number per $k$ for the entire library -- but the global Frobenius
error hides real spread across images. At the 95% milestone some crops reconstruct far
better than others. The histogram below is the *per-image* relative error at $k=k_{95}$:
the distribution the single global number summarizes.

In [ ]:
k_dist = k95
recon_k = reconstruct_with_k(X_centered, vt, k_dist)
per_image_err = (np.linalg.norm(X_centered - recon_k, axis=1)
                 / np.linalg.norm(X_centered, axis=1))
global_err = global_frobenius_error(X_centered, recon_k)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(per_image_err, bins=30, color="0.6", edgecolor="white")
ax.axvline(global_err, color="C3", linestyle="--", linewidth=1.5,
           label=f"global Frobenius = {global_err:.3f}")
ax.axvline(per_image_err.mean(), color="C0", linestyle=":", linewidth=1.5,
           label=f"mean per-image = {per_image_err.mean():.3f}")
ax.set_xlabel(f"Per-image relative L2 reconstruction error at k={k_dist}")
ax.set_ylabel("Number of images")
ax.set_title(f"Reconstruction quality varies across images (k={k_dist}, 95% variance)")
ax.legend(fontsize=8)
fig;
print(f"Per-image relative error at k={k_dist}: "
      f"median {np.median(per_image_err):.3f}, "
      f"90th pct {np.quantile(per_image_err, 0.9):.3f}, "
      f"worst {per_image_err.max():.3f}  "
      f"(global Frobenius {global_err:.3f}, mean per-image {per_image_err.mean():.3f})")

A visual confirmation: the same cell crop reconstructed from an increasing
number of modes. With only a few eigen-images it is a smudge; it takes on the
order of a hundred modes to sharpen on this real library, and past the 99%
variance milestone the extra modes change little.

In [ ]:
sample_idx = 0
ks_to_show = sorted({1, 8, k95, k99, n_modes})
fig, axes = plt.subplots(1, len(ks_to_show) + 1, figsize=(2.0 * (len(ks_to_show) + 1), 2.2))
axes[0].imshow(images[sample_idx], cmap="gray_r")
axes[0].set_title("original", fontsize=9)
axes[0].set_xticks([]); axes[0].set_yticks([])
for ax, k in zip(axes[1:], ks_to_show):
    recon = reconstruct_with_k(X_centered, vt, k)[sample_idx] + mean_image
    ax.imshow(recon.reshape(img_h, img_w), cmap="gray_r")
    ax.set_title(f"k={k}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"Reconstructing one blood-cell crop ({class_label(y[sample_idx])}) from k eigen-images")
fig;

### Recognition on the held-out test set

The payoff: classification in the compact basis, scored honestly. BloodMNIST ships an
official train / test split, so we use it rather than re-splitting the training images: we
learn the eigen-basis, the mean image, and the mode count from the **training** partition
alone, then classify each image of the **official test** partition by its nearest training
neighbour. No preprocessing, basis, mode count, or classifier ever sees the test set.

One caveat: this is an honest held-out estimate, not a "leakage-free" one. We drop the few
test crops that are pixel-identical to training ones, but with no donor identifiers we cannot
rule out same-donor near-duplicates across the split.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Guard the official boundary: DROP any test crop byte-identical to a training crop, so the
# held-out score is clean of the exact-duplicate leakage we can detect. (Same-donor NEAR-
# duplicates we cannot detect -- BloodMNIST ships no donor ids -- the interpretation says so.)
train_rows = {row.tobytes() for row in X}
keep = np.array([row.tobytes() not in train_rows for row in X_test])
n_dup = int((~keep).sum())
X_test, y_test = X_test[keep], y_test[keep]
print(f"Training library: {X.shape[0]} images   Official test set: {keep.size} images")
print(f"Removed {n_dup} test crops byte-identical to a training crop; scoring on {X_test.shape[0]}.")

# The eigen-basis, mean, and mode count were all fit on the TRAINING partition (Section 2); k is 95% of the training-library variance, chosen without ever touching the test set.
k_class = k95
Z_train = (X - mean_image) @ vt[:k_class].T
Z_test = (X_test - mean_image) @ vt[:k_class].T   # TRAIN mean & basis applied to the test set
print(f"Classifying in a {k_class}-dimensional eigen-basis (down from {X.shape[1]} raw pixels).")

In [ ]:
knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(Z_train, y)
acc_eigen = knn.score(Z_test, y_test)

# Baseline: the same 1-NN rule on raw pixels, for an honest comparison.
knn_raw = KNeighborsClassifier(n_neighbors=1)
knn_raw.fit(X, y)
acc_raw = knn_raw.score(X_test, y_test)

print(f"1-NN accuracy in the {k_class}-mode eigen-basis : {acc_eigen:.3f}")
print(f"1-NN accuracy on raw {X.shape[1]} pixels (baseline)   : {acc_raw:.3f}")
print(f"Dimensionality reduction: {X.shape[1]} -> {k_class} "
      f"({100 * k_class / X.shape[1]:.0f}% of the features), "
      f"accuracy change {acc_eigen - acc_raw:+.3f}")

The eigen-basis accuracy essentially matches the raw-pixel baseline -- and that is the
*point*, not a disappointment. PCA chooses its axes to capture **variance**, which is not
the same as **class separation**: the leading modes describe how the images vary, not how
the cell types differ, so a compact PCA basis *preserves* the classification signal rather
than sharpening it. A big accuracy jump here would actually be suspicious. When the goal is
classification *and* labels are available, the right move is to project onto
class-discriminating axes instead -- linear discriminant analysis -- which Section 4 below takes up
as the supervised complement to PCA. Here PCA earns its keep by **compression**, not by beating raw pixels.

The confusion matrix shows *where* the recognizer struggles. Rather than assume a pattern,
we print the largest off-diagonal cells below and read them directly: any clustering among
cell types with similar *grayscale* morphology is a hypothesis to check against the matrix,
not a given -- and recall we discarded colour and staining cues when we converted to
grayscale, so the recognizer cannot be confusing cells on those. This class-resolved view is
exactly what a single accuracy number hides.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, knn.predict(Z_test), labels=classes)
fig, ax = plt.subplots(figsize=(6.8, 5.8))
im = ax.imshow(cm, cmap="Blues")
ax.set_xlabel("Predicted cell type")
ax.set_ylabel("True cell type")
ax.set_title(f"1-NN confusion matrix in the eigen-basis (k={k_class})")
ax.set_xticks(range(n_classes)); ax.set_yticks(range(n_classes))
ax.set_xticklabels([class_label(c) for c in classes], rotation=45, ha="right", fontsize=7)
ax.set_yticklabels([class_label(c) for c in classes], fontsize=7)
for i in range(n_classes):
    for j in range(n_classes):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    fontsize=7, color="0.2" if cm[i, j] < cm.max() / 2 else "white")
fig.colorbar(im, ax=ax, fraction=0.046, label="count")
fig;

# Show the largest off-diagonal confusions so the "where it struggles" reading is
# demonstrated, not asserted.
off = cm.copy()
np.fill_diagonal(off, 0)
flat = np.argsort(off.ravel())[::-1]
print("Largest confusions (true -> predicted : count):")
for f in flat[:5]:
    i, j = np.unravel_index(f, off.shape)
    if off[i, j] == 0:
        break
    print(f"  {class_label(classes[i])} -> {class_label(classes[j])} : {off[i, j]}")

**Why the held-out test set matters.** We fit the eigen-basis, the mean image, the number of
modes, and the classifier using only the training partition, then evaluated on the official
test partition, which none of that fitting had touched. The reported accuracy is therefore an
estimate of performance on *new* images, not a memorization score. Fitting the representation on the
full (pooled train-and-test) data can bias the test estimate, often optimistically, because
information from the test distribution enters the fitted pipeline -- though on any single
finite split a contaminated analysis can move either way.

## 3. Robust PCA -- low-rank signal plus sparse corruption

PCA finds the best low-rank subspace, but "best" here means least-squares, and least
squares has no defense against a few gross outliers: a handful of corrupted entries can
bend the leading singular vectors away from the real structure. Real single-cell matrices
carry exactly this kind of damage -- doublets, ambient-RNA contamination, and
dropout-inflated counts sit as *sparse gross corruption* on top of the PBMC structure from
Section 1.

Robust PCA refuses to choose between the signal and the corruption. It splits the observed
matrix into a **low-rank** part `L` (the shared structure) plus a **sparse** part `S` (the
scattered outliers), giving the corruption its own bucket instead of letting it distort `L`.

As with ICA in Module 5b (Part 2), real expression data has no known low-rank/sparse truth to grade against,
so we test on a fixture whose answer we built -- a rank-3 matrix plus 5% large spikes --
recover `L` and `S`, score the recovery, and contrast it with a plain rank-3 SVD of the same
matrix.


In [ ]:
from ddm4bio.methods.decomposition import rpca, svd_lowrank
from ddm4bio.methods.validation import reconstruction_error

# A fixture whose answer we know: a rank-3 signal plus 5% large sparse spikes.
rng = np.random.default_rng(0)
n_rows, n_cols, rank_true = 200, 120, 3
L_true = rng.standard_normal((n_rows, rank_true)) @ rng.standard_normal((rank_true, n_cols))
amp = 5.0 * np.abs(L_true).mean()
corrupt = rng.random((n_rows, n_cols)) < 0.05                 # which entries are corrupted
S_true = np.where(corrupt, amp * rng.choice([-1.0, 1.0], size=(n_rows, n_cols)), 0.0)
X_obs = L_true + S_true

# Robust PCA splits the observed matrix into low-rank L + sparse S.
L_hat, S_hat = rpca(X_obs)

rel_L = reconstruction_error(L_true, L_hat, kind="rel_l2")
svals = np.linalg.svd(L_hat, compute_uv=False)
eff_rank = int((svals > 1e-6 * svals[0]).sum())
found = np.abs(S_hat) > 0.5 * amp
tp = int((found & corrupt).sum())
fp = int((found & ~corrupt).sum())
fn = int((~found & corrupt).sum())
precision, recall = tp / (tp + fp), tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)

# Contrast: a plain rank-3 SVD has to absorb the spikes into its factors.
U, s3, Vt = svd_lowrank(X_obs, rank_true)
rel_L_svd = reconstruction_error(L_true, (U * s3) @ Vt, kind="rel_l2")

print(f"corrupted entries : {int(corrupt.sum())} of {n_rows * n_cols} ({100 * corrupt.mean():.1f}%)")
print(f"robust PCA        : rel. L error {rel_L:.1e} | recovered rank {eff_rank} | spike F1 {f1:.2f}")
print(f"plain SVD (rank 3): rel. L error {rel_L_svd:.2f}")


In [ ]:
import matplotlib.pyplot as plt

vmax = np.abs(X_obs).max()
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2), constrained_layout=True)
panels = [
    (X_obs, "Observed  X = L + S"),
    (L_true, "True low-rank  L"),
    (L_hat, f"Recovered  L_hat  (err {rel_L:.0e})"),
    (S_hat, f"Recovered sparse  S_hat  (F1 {f1:.2f})"),
]
for ax, (mat, title) in zip(axes, panels):
    im = ax.imshow(mat, cmap="coolwarm", vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_title(title, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axes, shrink=0.85, label="value")
plt.show()


The split is exact where it counts: robust PCA recovers the true low-rank part to a relative
error of **1.2e-07** at the correct rank 3 and flags every injected spike (F1 **1.00**),
while a plain rank-3 SVD of the same matrix -- forced to explain the outliers with its
factors -- lands **17%** off the truth. That extra low-rank-plus-sparse *assumption* is the
whole difference: robust PCA still learns `L` and `S` from the data alone, but by asserting a
little more structure than bare PCA it can quarantine corruption that would otherwise bend the
components. It sits one rung up this module's ladder of asserted structure -- PCA (low-rank) ->
robust PCA (low-rank + sparse) -> ICA (independence, in Part 2 / Module 5b) -- and, as always, the synthetic score
verifies the unmixing recovers known sources; the messy-biology result still needs its own validation.


## 4. LDA: supervised projection for class separation

Every method so far has been *unsupervised*: PCA and robust PCA find structure
without ever seeing a label. But sometimes you *do* have labels -- healthy vs disease,
one cell type vs another -- and the question changes from "where is the variance?" to
"which directions best *separate the classes*?" Those are not the same question. Section 2
flagged exactly this: PCA on blood cells *preserved* the class signal but did not
*sharpen* it, because the leading modes describe how the images vary, not how the cell
types differ.

**Linear discriminant analysis (LDA)** answers the class-separation question directly.
Given labels, it finds the linear projection that maximizes between-class scatter
relative to within-class scatter -- the axis along which the class means are far apart
and each class is tight. It is the supervised complement to PCA, the tool Fisher
introduced for this problem. We validate it the usual way: on a fixture where the
class-separating direction is deliberately *not* the direction of most variance, so PCA
and LDA must disagree.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# A fixture that pits variance against class separation: the class signal lives on a
# LOW-variance axis (axis 0), buried under a HIGH-variance nuisance direction (axis 1).
rng = np.random.default_rng(0)
n, d = 600, 10
true_axis = np.zeros(d)
true_axis[0] = 1.0                                    # the direction that separates the classes
labels = rng.integers(0, 2, n)
Xf = rng.standard_normal((n, d)) * 0.6                # isotropic background noise
Xf[:, 1] += rng.standard_normal(n) * 4.0             # large nuisance variance, unrelated to class
Xf[:, 0] += (labels * 2.0 - 1.0) * 0.9               # a modest class shift along axis 0

pca_axis = PCA(n_components=2).fit(Xf).components_[0]
lda = LinearDiscriminantAnalysis().fit(Xf, labels)
lda_axis = lda.coef_[0] / np.linalg.norm(lda.coef_[0])
print(f"PCA top component vs the true class axis:   |cos| = {abs(pca_axis @ true_axis):.2f}  (chases the nuisance)")
print(f"LDA discriminant axis vs the true class axis: |cos| = {abs(lda_axis @ true_axis):.2f}  (recovers it)")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, proj, title in [(axes[0], Xf @ pca_axis, "PCA top component (unsupervised)"),
                        (axes[1], Xf @ lda_axis, "LDA discriminant axis (supervised)")]:
    for cls, color in [(0, "C0"), (1, "C3")]:
        ax.hist(proj[labels == cls], bins=30, alpha=0.6, color=color, label=f"class {cls}")
    ax.set_title(title)
    ax.set_xlabel("projection value")
axes[0].legend(fontsize=8)
axes[0].set_ylabel("count")
fig.suptitle("Same data, two 1-D projections: PCA mixes the classes, LDA separates them")
fig;

### Back to the blood cells

Section 2 above reduced BloodMNIST cells with an eigen-basis (PCA) and found it barely beat the
raw-pixel baseline at telling the eight cell types apart -- exactly because PCA optimizes
for variance, not class separation. Now we have the labels, so we can ask LDA for the
projection that *does* separate the types, and compare the two 2-D views head to head with
a nearest-neighbor classifier scored on each projection.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

from ddm4bio.datasets import get_dataset

bmn = get_dataset("bloodmnist")


def _gray(a):
    return a.mean(axis=-1).reshape(a.shape[0], -1).astype(float)


Xb_tr, yb_tr = _gray(bmn.payload["train_images"]), bmn.payload["train_labels"].ravel()
Xb_te, yb_te = _gray(bmn.payload["test_images"]), bmn.payload["test_labels"].ravel()

test_proj = {}
for name, model in [("PCA", PCA(n_components=2)),
                    ("LDA", LinearDiscriminantAnalysis(n_components=2))]:
    Ztr = model.fit_transform(Xb_tr, yb_tr) if name == "LDA" else model.fit_transform(Xb_tr)
    test_proj[name] = model.transform(Xb_te)
    acc = KNeighborsClassifier(n_neighbors=15).fit(Ztr, yb_tr).score(test_proj[name], yb_te)
    print(f"{name}(2) -> 15-NN test accuracy on the 2-D projection: {acc:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, name in zip(axes, ["PCA", "LDA"]):
    ax.scatter(test_proj[name][:, 0], test_proj[name][:, 1], c=yb_te, s=6, cmap="tab10", alpha=0.6)
    ax.set_title(f"{name}(2) projection of held-out cells")
    ax.set_xlabel("component 1")
    ax.set_ylabel("component 2")
fig.suptitle(f"BloodMNIST ({bmn.source}): PCA vs LDA 2-D projection, colored by cell type")
fig;

QC note. On the fixture LDA is decisive -- it recovers the class-separating axis (|cos| ~ 1)
that PCA, optimizing variance, misses entirely. On the real blood cells the gain is real but
*modest* (LDA's 2-D view edges PCA's), and that is the honest lesson: **LDA wins big exactly
when the discriminating signal is not the dominant variance; when class differences already
drive the variance, PCA is competitive.** Two caveats keep LDA in its lane: it assumes roughly
Gaussian classes with a shared covariance, and it yields at most (number of classes - 1)
discriminant axes -- so it is the supervised *complement* to PCA, not a universal upgrade.

## 5. Interpretation

Every ddm4bio analysis closes with an explicit claim and its named limitations. We close Part 1 on
its most quantifiable result -- recognizing cells from the compact eigen-cell basis.

In [ ]:
from ddm4bio.interpret import interpretation_block, show_interpretation

acc_se = float(np.sqrt(acc_eigen * (1.0 - acc_eigen) / y_test.size))
acc_lo, acc_hi = acc_eigen - 1.96 * acc_se, acc_eigen + 1.96 * acc_se

# Scoped to THIS section's claim (the classifier): one plain sentence plus the caveats that
# bear on it, folded into the limitations. The scree-tail, storage, and compression points
# live in Section 2 where they were made -- not re-dumped here.
block = interpretation_block(
    claim=(
        f"In a {k_class}-mode eigen-basis ({100 * k_class / X.shape[1]:.0f}% of the pixels), a "
        f"1-nearest-neighbour classifier matches raw-pixel accuracy on the held-out test set "
        f"({acc_eigen:.2f} vs {acc_raw:.2f}): the compact representation keeps the recognition "
        "signal rather than sharpening or losing it."
    ),
    limitations_list=[
        f"The held-out estimate (95% CI [{acc_lo:.2f}, {acc_hi:.2f}], n={y_test.size}) may be "
        f"mildly optimistic: we removed {n_dup} test crops byte-identical to training ones, but "
        "with no donor identifiers, same-donor near-duplicates across the split can't be excluded.",
        "Grayscale only -- we discarded the colour and staining cues that real blood-cell "
        "typing relies on.",
        "1-NN is a deliberately simple recognizer chosen for transparency, not a strong one.",
    ],
)
show_interpretation(block)

## Exercises

Your graded work for this module is **Problem Set 5 (PS5)**, distributed and
auto-graded through GitHub Classroom. This lesson reads a scree curve and
*eyeballs* where genuine structure fades into the noise tail. PS5 replaces that
judgement call with a principled significance test. Building on this lesson, it
asks you to:

- Build **Horn's parallel analysis**: permute each feature (gene) column
  independently to construct a rank-matched *noise* null of the eigenvalue
  spectrum, then keep only the leading principal components whose real eigenvalue
  beats the null -- recovering the latent dimensionality with a statistical test,
  not an eyeballed elbow.
- Validate it against synthetic matrices of **known planted rank**: show the test
  recovers the injected number of components exactly across a noise sweep, and
  degrades honestly only at extreme SNR.
- Contrast parallel analysis against the naive analytic **Marchenko-Pastur edge**,
  and show why the analytic shortcut over-counts on real, non-Gaussian expression
  data (real PBMC3k) even though the two rules agree on clean Gaussian noise.
- Write an interpretation block for the result using
  `ddm4bio.interpret.interpretation_block`.

Refer to the [PS5 repository README](https://github.com/symbiont-ai/ddm4bio/tree/main/problem_sets/ps5_dimreduction) for the submission and auto-grading details.

A lighter **companion set, Problem Set 1 (PS1)**, turns Section 2's eigen-basis into a *model of a normal cell*: reusing the same `eigen_basis` / `project` / `reconstruct` primitives, it denoises noisy microscopy by low-rank projection and flags out-of-QC images by their reconstruction error, on real BloodMNIST. See the [PS1 repository README](https://github.com/symbiont-ai/ddm4bio/tree/main/problem_sets/ps1_eigen_recognition).